# 03e | Window functions

Window functions are the centerpiece of SQL in data science interviews, and the line between "knows some SQL" and "fluent".

You have seen the problems they solve in pandas: ranking within groups, top N per group, cumulative sums, the previous row. They correspond to `rank`, `cumsum`, `shift` and `transform` from 02c.

**After this page you can:** write top-N per group, running totals, shares within a group, and comparisons with the previous row. About 55 minutes.

## 1. How they differ from GROUP BY

`GROUP BY` **collapses each group to one row**. A window function **keeps every row** and just adds a column with a value computed per group (exactly pandas' `transform`).

```sql
func() OVER (PARTITION BY group_col ORDER BY sort_col)
```

- `PARTITION BY`: what to group by (omit it and the whole table is one group);
- `ORDER BY`: the order within each group (needed for ranks and running totals).

In [ ]:
SELECT name, dept, salary,
       AVG(salary) OVER (PARTITION BY dept) AS dept_avg
FROM employees;

## 2. Three ranking functions

| Function | How ties are handled | Example: 150, 120, 120, 95 |
|---|---|---|
| `ROW_NUMBER()` | Ignores ties, numbers every row | 1, 2, 3, 4 |
| `RANK()` | Ties share a rank, then **skips** | 1, 2, 2, 4 |
| `DENSE_RANK()` | Ties share a rank, **no gaps** | 1, 2, 2, 3 |

This difference is a very common verbal question. Run the query below and compare:

In [ ]:
SELECT name, dept, salary,
       ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
       RANK()       OVER (PARTITION BY dept ORDER BY salary DESC) AS rnk,
       DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dense_rnk
FROM employees
ORDER BY dept, salary DESC;

## 3. Top N per group (a must-know)

You **cannot use a window function's result directly in `WHERE`** (in execution order, `WHERE` runs before window functions). The standard pattern has two steps:

1. Compute the rank in a CTE;
2. Filter on the rank in the outer query.

```sql
WITH ranked AS (
  SELECT *,
         ROW_NUMBER() OVER (PARTITION BY group_col ORDER BY sort_col DESC) AS rn
  FROM table_name
)
SELECT * FROM ranked WHERE rn <= N;
```

Which ranking function to use depends on how the problem treats ties: for exactly one row per group use `ROW_NUMBER`; to keep all ties use `RANK` or `DENSE_RANK`. **Ask how ties should be handled before you start**; that alone earns you points.

### ✏️ Exercise 1 (fill in the blanks)

Find the **one** highest-paid person in each department (break ties by `name` alphabetically). Output `dept`, `name`, `salary`.

In [ ]:
WITH ranked AS (
  SELECT dept, name, salary,
         ROW_NUMBER() OVER (PARTITION BY ____ ORDER BY ____ DESC, name) AS rn
  FROM employees
)
SELECT dept, name, salary
FROM ranked
WHERE rn = ____;

<details>
<summary><b>Answer</b></summary>

```sql
WITH ranked AS (
  SELECT dept, name, salary,
         ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC, name) AS rn
  FROM employees
)
SELECT dept, name, salary
FROM ranked
WHERE rn = 1;
```

</details>

### ✏️ Exercise 2 (write it yourself)

Find the employees in the **top two salary levels** of each department: keep all ties, with no gaps in the ranks (that is, the highest and second-highest salary levels in each department). Output `dept`, `name`, `salary`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

"Keep all ties, no gaps" means `DENSE_RANK()`. Filter on `rk <= 2`. The result should have 8 rows.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH ranked AS (
  SELECT dept, name, salary,
         DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS rk
  FROM employees
)
SELECT dept, name, salary
FROM ranked
WHERE rk <= 2;
```

</details>

### ✏️ Exercise 3 (write it yourself)

Find each user's **largest order** (any status). Break ties by the earliest date. Output `user_id`, `order_id`, `amount`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

`ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY amount DESC, order_date)`, then `WHERE rn = 1` in the outer query.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
WITH ranked AS (
  SELECT user_id, order_id, amount,
         ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY amount DESC, order_date) AS rn
  FROM orders
)
SELECT user_id, order_id, amount
FROM ranked
WHERE rn = 1;
```

</details>

## 4. Aggregate functions as window functions

`SUM`, `AVG`, `COUNT` and `MAX` can all be followed by `OVER`.

**Without `ORDER BY`:** the statistic for the whole group is attached to every row; useful for shares within a group.

**With `ORDER BY`:** it becomes "accumulate from the group's first row up to the current row", i.e. a running total.

In [ ]:
SELECT user_id, order_date, amount,
       SUM(amount) OVER (PARTITION BY user_id)                     AS user_total,
       SUM(amount) OVER (PARTITION BY user_id ORDER BY order_date) AS running_total
FROM orders
WHERE user_id IN (1, 3)
ORDER BY user_id, order_date;

### ✏️ Exercise 4 (write it yourself)

Compute each employee's salary as a share of **their department's total salary**. Output `name`, `dept`, `share`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

`1.0 * salary / SUM(salary) OVER (PARTITION BY dept)`. Don't forget the `1.0`, or integer division gives 0.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT name, dept,
       1.0 * salary / SUM(salary) OVER (PARTITION BY dept) AS share
FROM employees;
```

</details>

### ✏️ Exercise 5 (write it yourself)

In date order, compute the **running total of sales from completed orders across the whole site**. Output `order_id`, `order_date`, `amount`, `running_total`, sorted by date ascending (same-day orders by `order_id`).

In [ ]:
-- Write your SQL here

<details>
<summary><b>Hint</b></summary>

There are no groups this time, so no `PARTITION BY` is needed: `SUM(amount) OVER (ORDER BY order_date, order_id)`.

</details>

<details>
<summary><b>Answer</b></summary>

```sql
SELECT order_id, order_date, amount,
       SUM(amount) OVER (ORDER BY order_date, order_id) AS running_total
FROM orders
WHERE status = 'completed'
ORDER BY order_date, order_id;
```

</details>

### Moving averages

"The average of the last 3 orders" requires specifying the window's range (the frame):

```sql
AVG(amount) OVER (ORDER BY order_date ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)
```

This means "from 2 rows before up to the current row". A 7-day rolling window is `6 PRECEDING` (assuming exactly one row per day).

In [ ]:
SELECT order_id, order_date, amount,
       AVG(amount) OVER (ORDER BY order_date, order_id
                         ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_avg_3
FROM orders
WHERE status = 'completed'
ORDER BY order_date, order_id
LIMIT 6;

## 5. LAG and LEAD: the previous and next row

`LAG(col)` gets the value from the previous row in the group (pandas' `shift(1)`); `LEAD(col)` gets the next row. When there is no previous row, the result is NULL.

Uses: period-over-period growth, the gap between two events, checking whether days are consecutive.

In [ ]:
SELECT user_id, order_date, amount,
       LAG(order_date) OVER (PARTITION BY user_id ORDER BY order_date) AS prev_date,
       LAG(amount)     OVER (PARTITION BY user_id ORDER BY order_date) AS prev_amount
FROM orders
WHERE user_id IN (1, 3)
ORDER BY user_id, order_date;

### ✏️ Exercise 6 (write it yourself)

For each user's orders, compute **how much more this order cost than the previous one** (`amount` minus the previous order's `amount`), in a column named `diff`. Each user's first order has `diff` = NULL. Output `order_id`, `user_id`, `amount`, `diff`.

In [ ]:
-- Write your SQL here

<details>
<summary><b>Answer</b></summary>

```sql
SELECT order_id, user_id, amount,
       amount - LAG(amount) OVER (PARTITION BY user_id ORDER BY order_date) AS diff
FROM orders;
```

</details>

**🗣 Say it out loud:** What is the difference between ROW_NUMBER, RANK and DENSE_RANK? When do you use each?

<details>
<summary><b>Reference answer</b></summary>

All three rank rows within a group; they differ in how ties are handled. `ROW_NUMBER` gives every row a unique consecutive number, with tied rows ordered by any further ORDER BY keys (otherwise arbitrarily); `RANK` gives ties the same rank and then skips (1, 2, 2, 4); `DENSE_RANK` gives ties the same rank with no gaps (1, 2, 2, 3). Use `ROW_NUMBER` when you need exactly one row per group (deduplication, latest record); `DENSE_RANK` for "Nth highest salary"; `RANK` for competition-style standings.

**Short version:** They differ in how ties are handled. ROW_NUMBER gives unique consecutive numbers; RANK gives ties the same rank and then skips; DENSE_RANK gives ties the same rank without gaps. I use ROW_NUMBER to pick exactly one row per group, and DENSE_RANK for 'N-th highest value' questions.

</details>

## Summary

- `func() OVER (PARTITION BY ... ORDER BY ...)` does not collapse rows;
- Top N per group: compute `ROW_NUMBER` / `DENSE_RANK` in a CTE, filter in the outer query;
- `SUM() OVER (PARTITION BY g)` is the group total; adding `ORDER BY` makes it a running total;
- `LAG` / `LEAD` get the previous / next row.

On to **03f**.